# Modelos Generativos y Datos Sintéticos

Guía completa sobre cómo generar datos sintéticos realistas: desde los fundamentos
matemáticos hasta implementaciones visuales con animaciones.

| # | Sección |
|---|--------|
| 1 | ¿Qué son los datos sintéticos y por qué importan? |
| 2 | GAN — Generative Adversarial Network |
| 3 | Variantes de GAN (WGAN, CGAN, DCGAN) |
| 4 | VAE — Variational Autoencoder |
| 5 | Normalizing Flows |
| 6 | Modelos de Difusión (DDPM) |
| 7 | Modelos para datos tabulares (CTGAN, Copula, SMOTE) |
| 8 | Comparación y métricas de evaluación |

---
# 1. Datos Sintéticos: Motivación y Panorama

## ¿Por qué datos sintéticos?

| Problema | Solución con datos sintéticos |
|----------|------------------------------|
| **Escasez de datos** | Aumentar datasets pequeños |
| **Privacidad (GDPR)** | Reemplazar datos sensibles |
| **Clases desbalanceadas** | Sobre-muestrear clases minoritarias |
| **Costo de etiquetado** | Generar datos etiquetados |
| **Datos imposibles** | Casos extremos, fallos raros |

## Taxonomía de modelos generativos

```
Modelos Generativos
├── Explícitos (modelan p(x) directamente)
│   ├── Autoregresivos     PixelCNN, GPT
│   ├── Normalizing Flows  RealNVP, Glow
│   └── VAE                ELBO, β-VAE
└── Implícitos (muestrean sin modelar p(x))
    ├── GAN                Vanilla, WGAN, StyleGAN
    └── Difusión           DDPM, Score-based, Stable Diffusion
```

## Mapa del espacio latente

Todos los modelos aprenden una función:

$$G: \mathcal{Z} \to \mathcal{X}$$

donde $\mathcal{Z}$ es un espacio latente simple (ej. $\mathcal{N}(0,I)$) y
$\mathcal{X}$ es el espacio de datos reales.
La diferencia está en **cómo se aprende** $G$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

np.random.seed(42)

# Visualización del concepto central: transformar ruido en datos
fig = plt.figure(figsize=(14, 5))
gs = GridSpec(1, 3, figure=fig, wspace=0.35)

# 1. Espacio latente (ruido gaussiano)
ax1 = fig.add_subplot(gs[0])
z = np.random.randn(500, 2)
ax1.scatter(z[:,0], z[:,1], s=8, c='#3498DB', alpha=0.6)
ax1.set_title(r'Espacio latente $z \sim \mathcal{N}(0,I)$', fontsize=12, fontweight='bold')
ax1.set_xlabel('$z_1$'); ax1.set_ylabel('$z_2$')
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

# Flecha de transformación
ax_mid = fig.add_subplot(gs[1])
ax_mid.set_xlim(0, 1); ax_mid.set_ylim(0, 1)
ax_mid.annotate('', xy=(0.9, 0.5), xytext=(0.1, 0.5),
                arrowprops=dict(arrowstyle='->', lw=3, color='#E74C3C'))
ax_mid.text(0.5, 0.62, '$G(z)$', fontsize=18, ha='center', fontweight='bold', color='#E74C3C')
ax_mid.text(0.5, 0.35, 'Modelo\nGenerativo', fontsize=11, ha='center', color='gray')
ax_mid.axis('off')

# 2. Datos generados (mezcla de gaussianas)
ax3 = fig.add_subplot(gs[2])
centers = [(2,2),(-2,2),(0,-2)]
colors_g = ['#E74C3C','#2ECC71','#F39C12']
for (cx,cy), color in zip(centers, colors_g):
    pts = np.random.randn(160, 2) * 0.45 + [cx, cy]
    ax3.scatter(pts[:,0], pts[:,1], s=12, c=color, alpha=0.7)
ax3.set_title(r'Datos generados $\hat{x} = G(z)$', fontsize=12, fontweight='bold')
ax3.set_xlabel('$x_1$'); ax3.set_ylabel('$x_2$')
ax3.set_aspect('equal')
ax3.grid(True, alpha=0.3)

plt.suptitle('El objetivo: aprender la transformación $z \\mapsto x$', fontsize=14, fontweight='bold')
plt.show()

---
# 2. GAN — Generative Adversarial Network

## 2.1 Intuición: el juego del falsificador

Goodfellow et al. (2014) propusieron entrenar **dos redes en competencia**:

```
Ruido z ──→ [ Generador G ] ──→ Datos falsos x̂
                                       │
Datos reales x ────────────────────────┤
                                       ↓
                              [ Discriminador D ]
                                       │
                              P(real) ∈ [0, 1]
```

- **Generador** $G$: aprende a crear datos que engañen a $D$
- **Discriminador** $D$: aprende a distinguir real de falso

## 2.2 Formulación matemática

El entrenamiento es un juego minimax de suma cero:

$$\min_G \max_D \; V(G, D) = \mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

### Solución óptima (Goodfellow, 2014)

**Teorema**: Para $G$ fijo, el discriminador óptimo es:

$$D^*(x) = \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)}$$

**Teorema**: El mínimo global del juego se alcanza cuando $p_g = p_{\text{data}}$,
y el valor mínimo es $-\log 4$.

En ese punto $D^*(x) = 1/2$ para todo $x$ — el discriminador no puede distinguir.

### Equivalencia con divergencia Jensen-Shannon

$$C(G) = -\log 4 + 2 \cdot \text{JSD}(p_{\text{data}} \| p_g)$$

$$\text{JSD}(p \| q) = \frac{1}{2} \text{KL}\!\left(p \Big\| \frac{p+q}{2}\right) + \frac{1}{2} \text{KL}\!\left(q \Big\| \frac{p+q}{2}\right) \geq 0$$

## 2.3 Algoritmo de entrenamiento

```
Para cada iteración:

  // Paso 1: Actualizar Discriminador (k pasos)
  Para i = 1..k:
    Muestrear minibatch real:  {x^(i)} ~ p_data
    Muestrear ruido:           {z^(i)} ~ p_z
    Actualizar D subiendo el gradiente:
      ∇_D [log D(x) + log(1 - D(G(z)))]

  // Paso 2: Actualizar Generador (1 paso)
  Muestrear ruido: {z^(i)} ~ p_z
  Actualizar G bajando el gradiente:
      ∇_G [log(1 - D(G(z)))]   ← en la práctica: max log D(G(z))
```

### Truco de no-saturación

El objetivo original $\min_G \log(1-D(G(z)))$ satura al inicio.
En la práctica se usa $\max_G \log D(G(z))$ — mismo punto fijo, mejores gradientes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

np.random.seed(0)

# ─── MLP con numpy ───────────────────────────────────────────────────────────
def sigmoid(x):    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))
def relu(x):       return np.maximum(0, x)
def relu_grad(x):  return (x > 0).astype(float)
def leaky_relu(x, a=0.2):      return np.where(x > 0, x, a*x)
def leaky_relu_grad(x, a=0.2): return np.where(x > 0, 1.0, a)

class MLP:
    def __init__(self, dims, lr=0.001, activation='leaky_relu'):
        self.lr = lr
        self.act = leaky_relu if activation == 'leaky_relu' else relu
        self.act_g = leaky_relu_grad if activation == 'leaky_relu' else relu_grad
        self.W, self.b = [], []
        for i in range(len(dims)-1):
            scale = np.sqrt(2.0 / dims[i])
            self.W.append(np.random.randn(dims[i], dims[i+1]) * scale)
            self.b.append(np.zeros((1, dims[i+1])))

    def forward(self, x):
        self._cache = [x]
        h = x
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = h @ W + b
            h = self.act(z) if i < len(self.W)-1 else sigmoid(z)
            self._cache.append(z)
            self._cache.append(h)
        return h

    def backward_disc(self, y_pred, y_true):
        # BCE loss gradient
        n = y_pred.shape[0]
        dL = (y_pred - y_true) / (n * (y_pred*(1-y_pred) + 1e-8))
        grads_W, grads_b = [], []
        h = dL
        cache_rev = self._cache[::-1]
        # Last layer: sigmoid
        z_last = cache_rev[1]
        sig = cache_rev[0]
        d = h * sig * (1 - sig)
        layer_idx = len(self.W) - 1
        for i in range(layer_idx, -1, -1):
            a_in = self._cache[2*i]   # activation before this layer
            z_in = self._cache[2*i+1] # pre-activation
            grads_W.insert(0, a_in.T @ d)
            grads_b.insert(0, d.sum(axis=0, keepdims=True))
            if i > 0:
                d_prev = d @ self.W[i].T
                z_prev = self._cache[2*(i-1)+1]
                d = d_prev * self.act_g(z_prev)
        for i, (gW, gb) in enumerate(zip(grads_W, grads_b)):
            self.W[i] -= self.lr * np.clip(gW, -1, 1)
            self.b[i] -= self.lr * np.clip(gb, -1, 1)

# ─── Datos reales: mezcla de 3 gaussianas en 2D ──────────────────────────────
def sample_real(n=128):
    centers = [np.array([2., 2.]), np.array([-2., 2.]), np.array([0., -2.5])]
    probs   = [0.4, 0.3, 0.3]
    idx = np.random.choice(3, n, p=probs)
    x = np.array([centers[i] + np.random.randn(2)*0.5 for i in idx])
    return x

# ─── GAN simplificado con actualización simultánea ───────────────────────────
class SimpleGAN:
    def __init__(self, z_dim=2, x_dim=2, hidden=64):
        self.z_dim = z_dim
        self.G = MLP([z_dim, hidden, hidden, x_dim], lr=3e-4, activation='leaky_relu')
        self.D = MLP([x_dim, hidden, hidden, 1],     lr=1e-3, activation='leaky_relu')

    def sample_z(self, n): return np.random.randn(n, self.z_dim)

    def generate(self, n):
        z = self.sample_z(n)
        return self.G.forward(z)

    def train_step(self, x_real, n_disc=2):
        n = x_real.shape[0]

        # --- Train Discriminator ---
        for _ in range(n_disc):
            z = self.sample_z(n)
            x_fake = self.G.forward(z)
            x_all = np.vstack([x_real, x_fake])
            y_all = np.vstack([np.ones((n,1)), np.zeros((n,1))])
            d_out = self.D.forward(x_all)
            self.D.backward_disc(d_out, y_all)

        # --- Train Generator ---
        z = self.sample_z(n)
        x_fake = self.G.forward(z)
        # Forward through D (with new fake)
        d_fake = self.D.forward(x_fake)
        # Generator loss: -log D(G(z))  → BCE with y=1
        # We update D's input gradient and propagate back to G
        n_f = x_fake.shape[0]
        # dL/d(x_fake): gradient of -log(D(G(z))) w.r.t x_fake
        eps = 1e-8
        dL_df = -1.0 / (d_fake + eps) / n_f  # shape (n,1)
        # sigmoid gradient at last D layer
        sig = self.D._cache[-1]  # D output
        d_sig = dL_df * sig * (1 - sig)
        # propagate through D layers to get gradient w.r.t. x_fake
        d = d_sig
        n_layers = len(self.D.W)
        for i in range(n_layers-1, -1, -1):
            a_in = self.D._cache[2*i]
            z_in = self.D._cache[2*i+1]
            if i > 0:
                d_prev = d @ self.D.W[i].T
                z_prev = self.D._cache[2*(i-1)+1]
                d = d_prev * leaky_relu_grad(z_prev)
            else:
                d_x = d @ self.D.W[i].T  # grad w.r.t. x_fake
        # Now backprop through G using d_x
        # G.forward was called above; _cache holds the trace
        d_h = d_x
        n_g = len(self.G.W)
        grads_W, grads_b = [], []
        for i in range(n_g-1, -1, -1):
            a_in = self.G._cache[2*i]
            if i > 0:
                d_prev = d_h @ self.G.W[i].T
                z_prev = self.G._cache[2*(i-1)+1]
                d_h_new = d_prev * leaky_relu_grad(z_prev)
            else:
                d_h_new = d_h @ self.G.W[i].T
            grads_W.insert(0, a_in.T @ d_h)
            grads_b.insert(0, d_h.sum(axis=0, keepdims=True))
            d_h = d_h_new
        for i, (gW, gb) in enumerate(zip(grads_W, grads_b)):
            self.G.W[i] -= 3e-4 * np.clip(gW, -1, 1)
            self.G.b[i] -= 3e-4 * np.clip(gb, -1, 1)

        d_loss = -np.mean(np.log(d_out[:n]+1e-8) + np.log(1 - d_out[n:]+1e-8))
        g_loss = -np.mean(np.log(d_fake + 1e-8))
        return d_loss, g_loss

# ─── Entrenar y guardar snapshots ────────────────────────────────────────────
gan = SimpleGAN(z_dim=2, hidden=64)
snapshots = []
losses_D, losses_G = [], []
n_epochs = 800

for epoch in range(n_epochs):
    x_real = sample_real(256)
    dL, gL = gan.train_step(x_real, n_disc=2)
    losses_D.append(dL)
    losses_G.append(gL)
    if epoch % 40 == 0:
        x_fake = gan.generate(400)
        snapshots.append((epoch, x_fake.copy(), dL, gL))

print(f'Entrenamiento completado: {n_epochs} épocas')
print(f'Loss D final: {losses_D[-1]:.4f}  |  Loss G final: {losses_G[-1]:.4f}')

## 2.4 Visualización del entrenamiento

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Mostrar snapshots del entrenamiento
n_snaps = min(8, len(snapshots))
idxs = np.linspace(0, len(snapshots)-1, n_snaps, dtype=int)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# Datos de referencia
x_ref = sample_real(600)

for plot_i, snap_i in enumerate(idxs):
    epoch, x_fake, dL, gL = snapshots[snap_i]
    ax = axes[plot_i]
    ax.scatter(x_ref[:,0],  x_ref[:,1],  s=8, c='#3498DB', alpha=0.4, label='Real')
    ax.scatter(x_fake[:,0], x_fake[:,1], s=8, c='#E74C3C', alpha=0.6, label='Falso')
    ax.set_title(f'Época {epoch}\nD={dL:.2f}  G={gL:.2f}', fontsize=10, fontweight='bold')
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
    if plot_i == 0:
        ax.legend(fontsize=9, loc='upper right')

plt.suptitle('Evolución del GAN: Azul=Real, Rojo=Generado', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Curvas de pérdida
fig2, ax2 = plt.subplots(figsize=(12, 4))
window = 20
smooth_D = np.convolve(losses_D, np.ones(window)/window, mode='valid')
smooth_G = np.convolve(losses_G, np.ones(window)/window, mode='valid')
ax2.plot(smooth_D, color='#3498DB', linewidth=2, label='Loss Discriminador')
ax2.plot(smooth_G, color='#E74C3C', linewidth=2, label='Loss Generador')
ax2.axhline(y=np.log(2), color='gray', linestyle='--', alpha=0.7, label='log(2) ≈ Nash equilibrium')
ax2.set_xlabel('Época', fontsize=12)
ax2.set_ylabel('Pérdida BCE', fontsize=12)
ax2.set_title('Curvas de entrenamiento GAN', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2.5 Animación: GAN aprendiendo la distribución

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x_ref_anim = sample_real(500)

scat_real = axes[0].scatter([], [], s=10, c='#3498DB', alpha=0.5, label='Real')
scat_fake = axes[0].scatter([], [], s=10, c='#E74C3C', alpha=0.7, label='Generado')
axes[0].set_xlim(-5, 5); axes[0].set_ylim(-5, 5)
axes[0].set_aspect('equal')
axes[0].legend(fontsize=10, loc='upper right')
axes[0].grid(True, alpha=0.2)
title0 = axes[0].set_title('Época 0', fontsize=12, fontweight='bold')

line_D, = axes[1].plot([], [], '#3498DB', linewidth=2, label='Loss D')
line_G, = axes[1].plot([], [], '#E74C3C', linewidth=2, label='Loss G')
axes[1].axhline(y=np.log(2), color='gray', linestyle='--', alpha=0.5, label='Equilibrio')
axes[1].set_xlim(0, n_epochs); axes[1].set_ylim(0, max(max(losses_D[:50]), max(losses_G[:50]))*1.2)
axes[1].set_xlabel('Época', fontsize=11); axes[1].set_ylabel('Loss', fontsize=11)
axes[1].set_title('Pérdidas durante entrenamiento', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.2)

def init():
    scat_real.set_offsets(x_ref_anim)
    scat_fake.set_offsets(np.zeros((1,2)))
    line_D.set_data([], [])
    line_G.set_data([], [])
    return scat_real, scat_fake, line_D, line_G

def update(frame):
    epoch, x_fake, dL, gL = snapshots[frame]
    scat_fake.set_offsets(x_fake)
    title0.set_text(f'Época {epoch}  |  D={dL:.3f}  G={gL:.3f}')
    end = min(epoch+1, len(losses_D))
    line_D.set_data(range(end), losses_D[:end])
    line_G.set_data(range(end), losses_G[:end])
    # Adjust y-axis dynamically
    if end > 5:
        axes[1].set_ylim(0, max(max(losses_D[:end][-100:]), max(losses_G[:end][-100:])) * 1.3)
    return scat_real, scat_fake, line_D, line_G

anim = FuncAnimation(fig, update, init_func=init, frames=len(snapshots),
                     interval=350, blit=False)
plt.tight_layout()
plt.close()
HTML(anim.to_jshtml())

---
# 3. Variantes de GAN

## 3.1 Problemas del GAN vanilla

| Problema | Descripción |
|----------|-------------|
| **Mode collapse** | El generador solo aprende unos pocos modos de la distribución |
| **Inestabilidad** | Las pérdidas oscilan, difícil saber cuándo parar |
| **Vanishing gradient** | Cuando D es muy bueno, G no recibe señal útil |
| **Sin métrica de calidad** | La pérdida BCE no correlaciona con calidad visual |

## 3.2 WGAN — Wasserstein GAN (Arjovsky et al., 2017)

Reemplaza la divergencia Jensen-Shannon por la **distancia de Wasserstein-1** (Earth Mover's Distance):

$$W(p_{\text{data}}, p_g) = \inf_{\gamma \in \Pi} \mathbb{E}_{(x,y)\sim\gamma}[\|x - y\|]$$

Usando la dualidad de Kantorovich-Rubinstein:

$$W(p, q) = \sup_{\|f\|_L \leq 1} \mathbb{E}_{x\sim p}[f(x)] - \mathbb{E}_{x\sim q}[f(x)]$$

donde $f$ es una función 1-Lipschitz. El discriminador pasa a llamarse **crítico** y
no tiene sigmoide en la salida.

**Ventajas WGAN**:
- Pérdida correlaciona con calidad de las muestras
- Gradientes significativos incluso cuando D es bueno
- Elimina casi por completo el mode collapse

**WGAN-GP** (Gulrajani et al., 2017) impone la restricción Lipschitz con
un término de penalización de gradiente:

$$\mathcal{L}_D = \mathbb{E}[D(\tilde{x})] - \mathbb{E}[D(x)] + \lambda \mathbb{E}\left[(\|\nabla D(\hat{x})\|_2 - 1)^2\right]$$

## 3.3 Conditional GAN (cGAN, Mirza et al., 2014)

Condiciona tanto G como D en una etiqueta $y$:

$$\min_G \max_D V(G,D) = \mathbb{E}_{x\sim p_{\text{data}}}[\log D(x|y)] + \mathbb{E}_{z\sim p_z}[\log(1-D(G(z|y)|y))]$$

Permite **controlar qué tipo de dato generar** (ej. generar el dígito '7').

## 3.4 Otras variantes relevantes

| Variante | Contribución clave | Aplicación |
|----------|--------------------|------------|
| **DCGAN** | Arquitectura CNN para imágenes | Imágenes naturales |
| **StyleGAN 2/3** | Control de estilo y resolución 1024² | Caras fotorrealistas |
| **BigGAN** | Escala + proyección condicional | ImageNet class-conditional |
| **CycleGAN** | Traducción sin datos pareados | Horse↔Zebra, foto↔pintura |
| **CTGAN** | Modo de normalización condicional | **Datos tabulares** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Demostración visual: Mode Collapse vs WGAN
np.random.seed(7)

# Datos reales: anillo (distribución que GAN vanilla colapsa fácilmente)
def sample_ring(n=500, r=2.0, noise=0.15):
    theta = np.random.uniform(0, 2*np.pi, n)
    x = r * np.cos(theta) + np.random.randn(n) * noise
    y = r * np.sin(theta) + np.random.randn(n) * noise
    return np.column_stack([x, y])

# Simular mode collapse: generador que solo aprende 3 modos
def simulate_mode_collapse(n=500):
    modes = np.array([[2,0], [-1,1.7], [-1,-1.7]])  # 3 de 8 modos del anillo
    idx = np.random.choice(3, n)
    return modes[idx] + np.random.randn(n, 2) * 0.2

# Simular WGAN bien entrenado: cubre todos los modos
def simulate_wgan(n=500):
    return sample_ring(n, r=2.0, noise=0.22)

real = sample_ring(600)
collapsed = simulate_mode_collapse(600)
wgan_samples = simulate_wgan(600)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

titles = ['Distribución real\n(anillo)', 'GAN vanilla\n(mode collapse)', 'WGAN\n(cubre todos los modos)']
data   = [real, collapsed, wgan_samples]
colors = ['#3498DB', '#E74C3C', '#2ECC71']

for ax, title, pts, color in zip(axes, titles, data, colors):
    ax.scatter(pts[:,0], pts[:,1], s=10, c=color, alpha=0.5)
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

# Añadir flechas para indicar los modos colapsados
for mode in [[2,0], [-1,1.7], [-1,-1.7]]:
    axes[1].annotate('', xy=mode, xytext=[m*1.6 for m in mode],
                    arrowprops=dict(arrowstyle='->', color='darkred', lw=1.5))

plt.suptitle('Mode Collapse vs WGAN', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Conditional GAN: generar diferentes clases
np.random.seed(42)

def generate_conditional(n_per_class=200):
    classes = {
        0: {'center': [0, 3],    'cov': [[0.3,0],[0,0.3]], 'label': 'Clase A', 'color': '#E74C3C'},
        1: {'center': [-2.5,-1], 'cov': [[0.4,0.1],[0.1,0.2]], 'label': 'Clase B', 'color': '#3498DB'},
        2: {'center': [2.5,-1],  'cov': [[0.2,0],[0,0.5]], 'label': 'Clase C', 'color': '#2ECC71'},
    }
    real_pts, fake_pts, labels = [], [], []
    for cls, info in classes.items():
        r = np.random.multivariate_normal(info['center'], info['cov'], n_per_class)
        # Simular cGAN generando con leve desplazamiento (imperfección)
        noise = np.random.randn(n_per_class, 2) * 0.12
        f = np.random.multivariate_normal(info['center'], info['cov'], n_per_class) + noise
        real_pts.append((r, info['color'], info['label']))
        fake_pts.append((f, info['color']))

    return real_pts, fake_pts

real_pts, fake_pts = generate_conditional()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for pts, color, label in real_pts:
    axes[0].scatter(pts[:,0], pts[:,1], s=15, c=color, alpha=0.7, label=label)
axes[0].set_title('Datos reales (3 clases)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

for (fake, color), (real, _, label) in zip(fake_pts, real_pts):
    axes[1].scatter(fake[:,0], fake[:,1], s=15, c=color, alpha=0.7, marker='x',
                   label=f'{label} (generado)')
    # overlay real con transparencia
    axes[1].scatter(real[:,0], real[:,1], s=8, c=color, alpha=0.2)

axes[1].set_title('cGAN: datos generados por clase\n(× generado, ● real tenue)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

plt.suptitle('Conditional GAN: control sobre qué clase generar', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 4. VAE — Variational Autoencoder

## 4.1 Fundamento matemático

El VAE (Kingma & Welling, 2013) modela los datos con una variable latente:

$$p(x) = \int p(x|z)\, p(z)\, dz$$

donde $p(z) = \mathcal{N}(0, I)$ y $p(x|z)$ es el **decodificador**.

### El problema de la intractabilidad

El posterior real $p(z|x) = p(x|z)p(z)/p(x)$ es intratable.
Usamos una distribución variacional $q_\phi(z|x) \approx p(z|x)$.

### ELBO (Evidence Lower BOund)

Maximizar $\log p(x)$ equivale a maximizar la ELBO:

$$\log p(x) \geq \mathcal{L}(\theta, \phi; x) = \underbrace{\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]}_{\text{Reconstrucción}} - \underbrace{\text{KL}(q_\phi(z|x) \| p(z))}_{\text{Regularización}}$$

### Reparametrización

Para poder calcular gradientes a través del muestreo:

$$z = \mu_\phi(x) + \sigma_\phi(x) \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0,I)$$

La clave: el gradiente no pasa por la operación estocástica $\epsilon$.

### KL divergencia analítica

Cuando $q_\phi(z|x) = \mathcal{N}(\mu, \text{diag}(\sigma^2))$ y $p(z)=\mathcal{N}(0,I)$:

$$\text{KL}(q \| p) = \frac{1}{2}\sum_{j=1}^{d}\left(\mu_j^2 + \sigma_j^2 - \log\sigma_j^2 - 1\right)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

np.random.seed(1)

# ─── VAE implementado con numpy (2D → 1D latente → 2D) ───────────────────────
class VAE:
    def __init__(self, x_dim=2, z_dim=1, hidden=32, lr=2e-3):
        self.z_dim = z_dim
        self.lr = lr
        scale = 0.1
        # Encoder: x -> (mu, log_var)
        self.We1 = np.random.randn(x_dim, hidden)  * scale
        self.be1 = np.zeros(hidden)
        self.W_mu   = np.random.randn(hidden, z_dim) * scale
        self.b_mu   = np.zeros(z_dim)
        self.W_lv   = np.random.randn(hidden, z_dim) * scale
        self.b_lv   = np.zeros(z_dim)
        # Decoder: z -> x_hat
        self.Wd1 = np.random.randn(z_dim, hidden) * scale
        self.bd1 = np.zeros(hidden)
        self.Wd2 = np.random.randn(hidden, x_dim) * scale
        self.bd2 = np.zeros(x_dim)

    def encode(self, x):
        h = np.tanh(x @ self.We1 + self.be1)
        mu   = h @ self.W_mu + self.b_mu
        logv = h @ self.W_lv + self.b_lv
        return mu, logv, h

    def decode(self, z):
        h = np.tanh(z @ self.Wd1 + self.bd1)
        return h @ self.Wd2 + self.bd2, h

    def forward(self, x):
        mu, logv, h_enc = self.encode(x)
        eps = np.random.randn(*mu.shape)
        z   = mu + np.exp(0.5 * logv) * eps          # reparametrization
        x_hat, h_dec = self.decode(z)
        return x_hat, mu, logv, z, eps, h_enc, h_dec

    def loss(self, x, x_hat, mu, logv):
        recon = np.mean(np.sum((x - x_hat)**2, axis=1))    # MSE
        kl    = -0.5 * np.mean(np.sum(1 + logv - mu**2 - np.exp(logv), axis=1))
        return recon + kl, recon, kl

    def train_step(self, x, beta=1.0):
        x_hat, mu, logv, z, eps, h_enc, h_dec = self.forward(x)
        sigma = np.exp(0.5 * logv)
        n = x.shape[0]

        # Gradiente reconstrucción
        dL_dx_hat = 2*(x_hat - x) / n

        # Backprop decoder
        dWd2 = h_dec.T @ dL_dx_hat
        dbd2 = dL_dx_hat.sum(axis=0)
        dh_dec = dL_dx_hat @ self.Wd2.T * (1 - h_dec**2)
        dWd1 = z.T @ dh_dec
        dbd1 = dh_dec.sum(axis=0)
        dz   = dh_dec @ self.Wd1.T

        # Gradient through reparametrization
        dmu_recon   = dz
        dlogv_recon = dz * eps * 0.5 * sigma

        # KL gradient
        dmu_kl   = mu / n * beta
        dlogv_kl = 0.5 * (-1 + np.exp(logv)) / n * beta

        dmu   = dmu_recon   + dmu_kl
        dlogv = dlogv_recon + dlogv_kl

        # Backprop encoder
        dWmu = h_enc.T @ dmu
        dbmu = dmu.sum(axis=0)
        dWlv = h_enc.T @ dlogv
        dblv = dlogv.sum(axis=0)
        dh_enc = dmu @ self.W_mu.T + dlogv @ self.W_lv.T
        dh_enc *= (1 - h_enc**2)
        dWe1 = x.T @ dh_enc
        dbe1 = dh_enc.sum(axis=0)

        clip = 0.5
        for param, grad in [(self.Wd2,dWd2),(self.bd2,dbd2),(self.Wd1,dWd1),(self.bd1,dbd1),
                             (self.W_mu,dWmu),(self.b_mu,dbmu),(self.W_lv,dWlv),(self.b_lv,dblv),
                             (self.We1,dWe1),(self.be1,dbe1)]:
            param -= self.lr * np.clip(grad, -clip, clip)

        total, recon_l, kl_l = self.loss(x, x_hat, mu, logv)
        return total, recon_l, kl_l

    def generate(self, n):
        z = np.random.randn(n, self.z_dim)
        x_hat, _ = self.decode(z)
        return x_hat

# Datos: dos clases bien separadas
def sample_2class(n=300):
    c1 = np.random.randn(n//2, 2) * 0.4 + [1.5, 0]
    c2 = np.random.randn(n//2, 2) * 0.4 + [-1.5, 0]
    return np.vstack([c1, c2])

vae = VAE(x_dim=2, z_dim=1, hidden=48, lr=3e-3)
vae_snapshots = []
vae_losses, vae_kls, vae_recons = [], [], []

for epoch in range(1000):
    x_batch = sample_2class(256)
    tot, rc, kl = vae.train_step(x_batch, beta=1.0)
    vae_losses.append(tot)
    vae_kls.append(kl)
    vae_recons.append(rc)
    if epoch % 50 == 0:
        gen = vae.generate(400)
        vae_snapshots.append((epoch, gen.copy(), tot, rc, kl))

print(f'VAE entrenado: loss={vae_losses[-1]:.4f}  KL={vae_kls[-1]:.4f}  Recon={vae_recons[-1]:.4f}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualizar: espacio latente + generación
x_test = sample_2class(400)
mu_enc, logv_enc, _ = vae.encode(x_test)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Espacio latente aprendido
colors_test = ['#E74C3C']*200 + ['#3498DB']*200
scatter = axes[0].scatter(mu_enc[:,0], np.zeros_like(mu_enc[:,0]) + np.random.randn(400)*0.05,
                          c=colors_test, s=15, alpha=0.7)
axes[0].set_xlabel('z (espacio latente 1D)', fontsize=12)
axes[0].set_title('Espacio latente codificado\n(1D)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend(handles=[plt.Line2D([],[],marker='o',ls='',c='#E74C3C',label='Clase 1'),
                         plt.Line2D([],[],marker='o',ls='',c='#3498DB',label='Clase 2')], fontsize=10)

# 2. Datos generados vs reales
x_gen = vae.generate(400)
axes[1].scatter(x_test[:,0], x_test[:,1], s=10, c='#3498DB', alpha=0.4, label='Real')
axes[1].scatter(x_gen[:,0],  x_gen[:,1],  s=10, c='#E74C3C', alpha=0.7, label='VAE generado')
axes[1].set_title('Datos reales vs generados', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

# 3. Traversal del espacio latente
z_vals = np.linspace(-2.5, 2.5, 40).reshape(-1,1)
x_traversal, _ = vae.decode(z_vals)
scatter3 = axes[2].scatter(x_traversal[:,0], x_traversal[:,1],
                            c=z_vals[:,0], cmap='RdBu', s=30, alpha=0.9)
plt.colorbar(scatter3, ax=axes[2], label='z')
axes[2].set_title('Traversal del espacio latente\n(z de -2.5 a 2.5)', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].set_aspect('equal')

plt.suptitle('VAE: espacio latente, generación y traversal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Curvas de pérdida VAE
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 4))
sm = lambda s, w=20: np.convolve(s, np.ones(w)/w, 'valid')
axes2[0].plot(sm(vae_losses), '#9B59B6', lw=2, label='ELBO total')
axes2[0].set_title('Loss total (ELBO)', fontsize=12, fontweight='bold')
axes2[0].set_xlabel('Época'); axes2[0].grid(True, alpha=0.3)
axes2[1].plot(sm(vae_recons), '#E74C3C', lw=2, label='Reconstrucción')
axes2[1].plot(sm(vae_kls), '#3498DB', lw=2, label='KL divergencia')
axes2[1].set_title('Componentes ELBO', fontsize=12, fontweight='bold')
axes2[1].set_xlabel('Época'); axes2[1].legend(fontsize=10); axes2[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# 5. Normalizing Flows

## 5.1 Fundamento matemático

Un **Normalizing Flow** aprende una transformación invertible $f: \mathcal{Z} \to \mathcal{X}$
tal que $x = f(z)$ con $z \sim p_z$.

### Regla de cambio de variable

Si $z \sim p_z$ y $x = f(z)$:

$$p_x(x) = p_z(f^{-1}(x)) \left|\det\left(\frac{\partial f^{-1}}{\partial x}\right)\right|$$

### Log-verosimilitud exacta

$$\log p_x(x) = \log p_z(z) - \log\left|\det\left(J_f(z)\right)\right|$$

donde $J_f = \partial f / \partial z$ es el Jacobiano.

### Composición de transformaciones

$$x = f_K \circ f_{K-1} \circ \cdots \circ f_1(z)$$

$$\log p_x(x) = \log p_z(z) - \sum_{k=1}^{K} \log\left|\det J_{f_k}\right|$$

### Comparación con otros modelos

| Modelo | $p(x)$ exacta | Muestras rápidas | Espacio latente | Entrenamiento |
|--------|:-------------:|:----------------:|:---------------:|:--------------:|
| GAN | No | Sí | Sí | Adversarial |
| VAE | Aprox. (ELBO) | Sí | Sí | MLE + KL |
| **Flow** | **Sí** | **Sí** | Sí | **MLE exacta** |
| Difusión | Aprox. | Lenta | Implícito | Score matching |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

np.random.seed(5)

# ─── Planar Flow (Rezende & Mohamed, 2015) ───────────────────────────────────
# Transformación: f(z) = z + u * h(w^T z + b)
# donde h = tanh y la condición de invertibilidad: u^T w >= -1

class PlanarFlow:
    def __init__(self, dim=2, lr=1e-3):
        self.w = np.random.randn(dim, 1) * 0.1
        self.u = np.random.randn(dim, 1) * 0.1
        self.b = np.zeros(1)
        self.dim = dim
        self.lr = lr

    def _u_hat(self):
        # Garantiza invertibilidad: u_hat = u + (m(w^Tu) - w^Tu) * w / ||w||^2
        wtu = float(self.w.T @ self.u)
        m = -1 + np.log1p(np.exp(wtu))  # softplus - 1
        u_hat = self.u + (m - wtu) * self.w / (np.linalg.norm(self.w)**2 + 1e-8)
        return u_hat

    def forward(self, z):
        u_hat = self._u_hat()
        lin = z @ self.w + self.b       # (n,1)
        h = np.tanh(lin)                # (n,1)
        z_new = z + (u_hat * h).T       # (n, dim)  -- broadcasting trick
        # log|det J| = log|1 + u_hat^T psi(z)|
        psi = (1 - h**2) * self.w.T    # (n, dim)
        log_det = np.log(np.abs(1 + psi @ u_hat) + 1e-8)  # (n,1)
        return z_new, log_det.squeeze(), h, lin

class NormalizingFlow:
    def __init__(self, dim=2, K=8, lr=1e-3):
        self.flows = [PlanarFlow(dim, lr) for _ in range(K)]
        self.lr = lr

    def log_prob(self, x):
        z = x.copy()
        # We go backward: sample x, find z, compute log p(x)
        log_det_sum = np.zeros(x.shape[0])
        # Apply flows in reverse (for density estimation)
        # For simplicity, we'll use forward mode here (generative)
        return log_det_sum

    def sample(self, n):
        z = np.random.randn(n, 2)
        log_det_sum = np.zeros(n)
        zs = [z.copy()]
        for flow in self.flows:
            z, log_det, _, _ = flow.forward(z)
            log_det_sum += log_det
            zs.append(z.copy())
        return z, log_det_sum, zs

# Visualización de cómo cada capa transforma la distribución
nf = NormalizingFlow(dim=2, K=6, lr=1e-3)
# Inicializar con pesos que generen una transformación interesante
for i, flow in enumerate(nf.flows):
    angle = i * np.pi / 3
    flow.w = np.array([[np.cos(angle)], [np.sin(angle)]]) * 1.5
    flow.u = np.array([[np.sin(angle)], [-np.cos(angle)]]) * 2.0
    flow.b = np.array([i * 0.3])

z_samples = np.random.randn(1000, 2)
_, _, zs = nf.sample(1000)

# Visualizar las primeras 4 transformaciones
fig, axes = plt.subplots(1, min(4, len(zs)), figsize=(16, 4))
cmap_flow = plt.cm.viridis

for i, (ax, zi) in enumerate(zip(axes, zs[:4])):
    c_idx = np.sqrt(zi[:,0]**2 + zi[:,1]**2)  # color por distancia al origen
    ax.scatter(zi[:,0], zi[:,1], c=c_idx, cmap='plasma', s=8, alpha=0.7)
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.set_title(f'Capa {i}: z{i}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.2)

plt.suptitle('Normalizing Flow: transformación capa a capa de N(0,I)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Animación: Flujo de transformación capa a capa

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(7, 7))
n_pts = 800
z_base = np.random.randn(n_pts, 2)
_, _, all_zs = nf.sample(n_pts)

c_base = np.arctan2(z_base[:,1], z_base[:,0])  # color por ángulo (persistente)

scat = ax.scatter([], [], c=[], cmap='hsv', vmin=-np.pi, vmax=np.pi, s=12, alpha=0.8)
ax.set_xlim(-4.5, 4.5); ax.set_ylim(-4.5, 4.5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
title = ax.set_title('', fontsize=14, fontweight='bold')

# Intermediate frames with interpolation
def get_interp_frames(zs, steps_per_layer=15):
    frames = []
    for i in range(len(zs)-1):
        for t in np.linspace(0, 1, steps_per_layer):
            interp = (1-t)*zs[i] + t*zs[i+1]
            frames.append((interp, i, t))
    frames.append((zs[-1], len(zs)-1, 1.0))
    return frames

anim_frames = get_interp_frames(all_zs, steps_per_layer=12)

def update(frame_idx):
    pts, layer, t = anim_frames[frame_idx]
    scat.set_offsets(pts)
    scat.set_array(c_base)
    title.set_text(f'Normalizing Flow — Capa {layer}  (t={t:.2f})')
    return scat,

anim_nf = FuncAnimation(fig, update, frames=len(anim_frames), interval=60, blit=True)
plt.close()
HTML(anim_nf.to_jshtml())

---
# 6. Modelos de Difusión

## 6.1 Intuición

Los modelos de difusión (Ho et al., 2020 — DDPM) aprenden a **invertir el proceso
de añadir ruido** gradualmente.

```
Proceso forward (fijo):   x₀ ──→ x₁ ──→ x₂ ──→ ··· ──→ x_T ~ N(0,I)
                              +ε    +ε    +ε

Proceso reverse (aprendido): x_T ──→ ··· ──→ x₁ ──→ x₀
                                  -ε_θ  -ε_θ  -ε_θ
```

## 6.2 Proceso forward

Se define una cadena de Markov que añade ruido gaussiano gradualmente:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t;\ \sqrt{1-\beta_t}\, x_{t-1},\ \beta_t I)$$

Con el **truco de reparametrización**, podemos saltar directo al paso $t$:

$$q(x_t | x_0) = \mathcal{N}(x_t;\ \sqrt{\bar{\alpha}_t}\, x_0,\ (1-\bar{\alpha}_t) I)$$

donde $\alpha_t = 1-\beta_t$ y $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$.

## 6.3 Proceso reverse y objetivo

La red neuronal $\epsilon_\theta(x_t, t)$ aprende a predecir el ruido añadido:

$$\mathcal{L} = \mathbb{E}_{x_0, \epsilon, t}\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

## 6.4 Sampling (DDPM)

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}\epsilon_\theta(x_t,t)\right) + \sigma_t z$$

## 6.5 Ventajas sobre GAN y VAE

- **Sin adversarial training**: más estable, sin mode collapse
- **Log-likelihood exacta**: mediante ELBO
- **Calidad superior** en imágenes de alta resolución (2021+)
- **Desventaja**: sampling lento ($T$ pasos, típicamente 1000)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

np.random.seed(3)

# ─── DDPM simplificado en 2D para visualización ──────────────────────────────
T = 200            # pasos de difusión
beta_min, beta_max = 1e-4, 0.02
betas = np.linspace(beta_min, beta_max, T)
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

def q_sample(x0, t):
    # Muestra x_t dado x_0: x_t = sqrt(alpha_bar_t)*x0 + sqrt(1-alpha_bar_t)*eps
    abar = alpha_bars[t].reshape(-1, 1)
    eps  = np.random.randn(*x0.shape)
    xt   = np.sqrt(abar) * x0 + np.sqrt(1 - abar) * eps
    return xt, eps

def ddpm_reverse_step(xt, t, eps_pred):
    # Un paso del reverse process
    a_t   = alphas[t]
    ab_t  = alpha_bars[t]
    beta_t = betas[t]
    coef = (1 - a_t) / np.sqrt(1 - ab_t)
    mean = (xt - coef * eps_pred) / np.sqrt(a_t)
    if t > 0:
        z = np.random.randn(*xt.shape)
        return mean + np.sqrt(beta_t) * z
    return mean

# Red simplificada: predice epsilon como promedio ponderado
# En la práctica: UNet, Transformer
# Aquí usamos una 'red' trivial que usa el conocimiento de la data
class ToyDDPM:
    def __init__(self, data_mean, data_cov):
        self.mu = data_mean
        self.cov = data_cov

    def predict_noise(self, xt, t):
        # Predictor analítico del ruido (usando conocimiento de p_data)
        abar = alpha_bars[t]
        # Estimacion de x0 desde xt y la distribucion conocida
        # eps_pred ≈ (xt - sqrt(abar)*x0_hat) / sqrt(1-abar)
        # x0_hat = E[x0|xt] = abar*mu / (abar + (1-abar)) (aprox para gaussiana)
        ab = float(abar)
        x0_hat = (ab * self.mu + np.sqrt(ab*(1-ab)) * (xt - np.sqrt(ab)*self.mu) / (1-ab+1e-8))
        eps_pred = (xt - np.sqrt(ab) * x0_hat) / (np.sqrt(1-ab) + 1e-8)
        return np.clip(eps_pred, -3, 3)

# Datos de ejemplo
x0_real = sample_2class(300)
toy_model = ToyDDPM(x0_real.mean(0), np.cov(x0_real.T))

# Visualizar proceso forward
t_steps = [0, 20, 50, 100, 150, T-1]
x_sample = x0_real[:200].copy()

fig, axes = plt.subplots(1, len(t_steps), figsize=(17, 3.5))
for ax, t in zip(axes, t_steps):
    ts = np.full(len(x_sample), t)
    xt, _ = q_sample(x_sample, ts)
    ax.scatter(xt[:,0], xt[:,1], s=8, c='steelblue', alpha=0.6)
    abar = alpha_bars[t]
    ax.set_title(f't={t}\n$\\bar{{\\alpha}}$={abar:.3f}', fontsize=10, fontweight='bold')
    ax.set_xlim(-4,4); ax.set_ylim(-3,3); ax.set_aspect('equal'); ax.grid(True, alpha=0.2)

plt.suptitle('DDPM: Proceso forward — Añadiendo ruido gradualmente', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Animación: Proceso forward y reverse de difusión

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_pts = 300
x0_anim = sample_2class(n_pts)

# Forward frames
fwd_ts = list(range(0, T, 5))
fwd_frames = []
for t in fwd_ts:
    ts = np.full(n_pts, t)
    xt, _ = q_sample(x0_anim, ts)
    fwd_frames.append(('forward', t, xt.copy()))

# Reverse: sample from noise and denoise
xT = np.random.randn(n_pts, 2)
xt_rev = xT.copy()
rev_ts = list(range(T-1, -1, 5))
rev_frames = []
for t in rev_ts:
    rev_frames.append(('reverse', t, xt_rev.copy()))
    eps_pred = toy_model.predict_noise(xt_rev, t)
    xt_rev = ddpm_reverse_step(xt_rev, t, eps_pred)
rev_frames.append(('reverse', 0, xt_rev.copy()))

all_frames = fwd_frames + [('pause', 0, fwd_frames[-1][2])] * 5 + rev_frames

fig, ax = plt.subplots(figsize=(7, 6))
scat_diff = ax.scatter([], [], s=12, c='steelblue', alpha=0.7)
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
title_diff = ax.set_title('', fontsize=13, fontweight='bold')
stage_text = ax.text(0.02, 0.97, '', transform=ax.transAxes, fontsize=11,
                     va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

def update_diff(i):
    mode, t, pts = all_frames[i]
    scat_diff.set_offsets(pts)
    if mode == 'forward':
        scat_diff.set_color('#3498DB')
        title_diff.set_text(f'Forward: t={t}  (añadiendo ruido)')
        stage_text.set_text(f'$\\bar{{\\alpha}}_t$ = {alpha_bars[t]:.3f}')
    elif mode == 'reverse':
        scat_diff.set_color('#E74C3C')
        title_diff.set_text(f'Reverse: t={t}  (eliminando ruido)')
        stage_text.set_text(f'Denoising...')
    else:
        title_diff.set_text('← Forward completo | Reverse comienza →')
    return scat_diff,

anim_ddpm = FuncAnimation(fig, update_diff, frames=len(all_frames), interval=80, blit=False)
plt.close()
HTML(anim_ddpm.to_jshtml())

---
# 7. Modelos para Datos Tabulares

## 7.1 Desafíos de los datos tabulares

Los datos tabulares presentan retos únicos que los hacen más difíciles que imágenes:

| Desafío | Descripción |
|---------|-------------|
| **Tipos mixtos** | Numéricas continuas + categóricas ordinales/nominales |
| **Distribuciones multimodales** | Columnas con múltiples picos |
| **Dependencias complejas** | Correlaciones no lineales entre columnas |
| **Clases desbalanceadas** | Fraude, enfermedades raras |
| **Columnas con moda** | Muchos ceros, valores únicos |

## 7.2 CTGAN (Xu et al., 2019)

**Conditional Tabular GAN** aborda estos retos con:

1. **Mode-specific normalization**: normaliza columnas multimodales con BayesianGMM
2. **Conditional vector**: condiciona G en categorías muestreadas uniformemente
3. **Training-by-sampling**: asegura que G aprende todos los modos

## 7.3 Cópulas Gaussianas

Una **cópula** separa la estructura de dependencia de las distribuciones marginales:

$$F(x_1,\ldots,x_n) = C(F_1(x_1),\ldots,F_n(x_n))$$

donde $C$ es la cópula y $F_i$ son las marginales.

La **Cópula Gaussiana** usa correlaciones lineales en el espacio normal:

$$C^{Ga}_\Sigma(u) = \Phi_\Sigma(\Phi^{-1}(u_1),\ldots,\Phi^{-1}(u_n))$$

## 7.4 SMOTE (Chawla et al., 2002)

**Synthetic Minority Oversampling Technique**: interpola entre vecinos cercanos
de la clase minoritaria:

$$x_{\text{new}} = x_i + \lambda \cdot (x_{\hat{k}} - x_i), \quad \lambda \sim \mathcal{U}(0,1)$$

donde $x_{\hat{k}}$ es uno de los $k$ vecinos más cercanos de $x_i$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# ─── SMOTE desde cero ────────────────────────────────────────────────────────
def euclidean_dist(a, b):
    return np.sqrt(np.sum((a - b)**2, axis=1))

def smote(X_minority, n_synthetic, k=5):
    synthetic = []
    for _ in range(n_synthetic):
        # Elegir muestra aleatoria de la minoría
        idx = np.random.randint(0, len(X_minority))
        xi = X_minority[idx]
        # Encontrar k vecinos más cercanos
        dists = np.sqrt(np.sum((X_minority - xi)**2, axis=1))
        dists[idx] = np.inf  # excluir a sí mismo
        nn_idx = np.argsort(dists)[:k]
        # Elegir vecino aleatorio
        chosen = X_minority[np.random.choice(nn_idx)]
        # Interpolar
        lam = np.random.uniform(0, 1)
        x_new = xi + lam * (chosen - xi)
        synthetic.append(x_new)
    return np.array(synthetic)

# ─── Cópula Gaussiana desde cero ─────────────────────────────────────────────
def gaussian_copula_sample(n, marginals, corr_matrix):
    d = len(marginals)
    # 1. Muestrear de MVN con la correlación dada
    L = np.linalg.cholesky(corr_matrix)
    z = np.random.randn(n, d) @ L.T
    # 2. Transformar a uniforme via CDF normal
    u = stats.norm.cdf(z)
    # 3. Transformar cada marginal via PPF (inversa de CDF)
    samples = np.column_stack([marginal.ppf(u[:, i]) for i, marginal in enumerate(marginals)])
    return samples

# Dataset desbalanceado
n_majority = 300
n_minority = 30
X_maj = np.random.randn(n_majority, 2) * 1.2 + [1, 1]
X_min = np.random.randn(n_minority, 2) * 0.5 + [-2, -2]

# Aplicar SMOTE
X_smote = smote(X_min, n_synthetic=200, k=5)

# Dataset tabular con cópula gaussiana
# Columnas: Edad (normal), Ingreso (lognormal), Score (beta)
corr = np.array([[1.0, 0.7, 0.3],
                  [0.7, 1.0, 0.5],
                  [0.3, 0.5, 1.0]])
marginals_tabular = [
    stats.norm(loc=35, scale=10),        # Edad
    stats.lognorm(s=0.5, scale=np.exp(10.5)),  # Ingreso
    stats.beta(a=2, b=5),                # Score crediticio [0,1]
]
X_copula = gaussian_copula_sample(500, marginals_tabular, corr)

# ─── Visualización ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# SMOTE
axes[0].scatter(X_maj[:,0], X_maj[:,1], s=8,  c='#3498DB', alpha=0.4, label=f'Mayoría (n={n_majority})')
axes[0].scatter(X_min[:,0], X_min[:,1], s=25, c='#E74C3C', alpha=0.9, label=f'Minoría (n={n_minority})', zorder=3)
axes[0].scatter(X_smote[:,0], X_smote[:,1], s=12, c='#E74C3C', alpha=0.4, marker='x',
               label=f'SMOTE sintético (n={len(X_smote)})')
axes[0].set_title('SMOTE: oversampling minoría', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Cópula - scatter matrix (primeras 2 columnas)
axes[1].scatter(X_copula[:,0], X_copula[:,1], s=8, c='#9B59B6', alpha=0.5)
axes[1].set_xlabel('Edad', fontsize=11); axes[1].set_ylabel('Ingreso', fontsize=11)
axes[1].set_title('Cópula Gaussiana:\nEdad vs Ingreso (ρ=0.7)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Distribuciones marginales
axes[2].hist(X_copula[:,0], bins=30, alpha=0.7, color='#E74C3C', label='Edad')
ax2b = axes[2].twinx()
ax2b.hist(X_copula[:,2], bins=30, alpha=0.5, color='#2ECC71', label='Score crediticio')
axes[2].set_title('Marginales: Edad y Score\n(distribuciones distintas, correlacionadas)', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Valor'); axes[2].set_ylabel('Freq (Edad)', color='#E74C3C')
ax2b.set_ylabel('Freq (Score)', color='#2ECC71')
axes[2].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 7.5 KDE — Kernel Density Estimation

La KDE estima la densidad de probabilidad de forma no paramétrica:

$$\hat{p}(x) = \frac{1}{n h} \sum_{i=1}^{n} K\left(\frac{x - x_i}{h}\right)$$

donde $K$ es un kernel (ej. gaussiano) y $h$ el ancho de banda.

**Uso en datos sintéticos**: muestrear de $\hat{p}$ directamente.

**Limitación**: escala mal con dimensiones altas (*curse of dimensionality*).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

np.random.seed(10)

# KDE para generar datos sintéticos
# Datos originales: distribución bimodal
x_original = np.concatenate([
    np.random.randn(150) * 0.8 + 2,
    np.random.randn(100) * 0.5 - 1,
])

# Ajustar KDE
kde = gaussian_kde(x_original, bw_method='silverman')

# Comparar diferentes anchos de banda
x_grid = np.linspace(-4, 6, 300)
bws = [0.1, 0.5, 1.0, 'silverman']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Densidades con diferentes h
colors_kde = ['#E74C3C','#F39C12','#3498DB','#2ECC71']
for bw, color in zip(bws, colors_kde):
    kde_bw = gaussian_kde(x_original, bw_method=bw)
    label = f'h={bw}' if bw != 'silverman' else f'Silverman (h={kde_bw.factor:.2f})'
    axes[0].plot(x_grid, kde_bw(x_grid), color=color, linewidth=2, label=label)

axes[0].hist(x_original, bins=30, density=True, alpha=0.3, color='gray', label='Histograma')
axes[0].set_title('KDE con diferentes anchos de banda', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel('x'); axes[0].set_ylabel('Densidad')

# Muestrear de la KDE (datos sintéticos)
n_synthetic_kde = 400
x_synthetic_kde = kde.resample(n_synthetic_kde).flatten()

axes[1].hist(x_original, bins=25, density=True, alpha=0.5, color='#3498DB', label=f'Original (n={len(x_original)})')
axes[1].hist(x_synthetic_kde, bins=25, density=True, alpha=0.5, color='#E74C3C', label=f'KDE sintético (n={n_synthetic_kde})')
axes[1].plot(x_grid, kde(x_grid), 'k-', linewidth=2, label='KDE estimada')
axes[1].set_title('Datos reales vs sintéticos (KDE)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel('x'); axes[1].set_ylabel('Densidad')

plt.tight_layout()
plt.show()

---
# 8. Comparación de Modelos y Métricas de Evaluación

## 8.1 Métricas de calidad para datos sintéticos

Evaluar datos sintéticos es un problema abierto. Las métricas principales son:

### Para imágenes

| Métrica | Fórmula / Idea | Mide |
|---------|----------------|------|
| **FID** (Fréchet Inception Distance) | Dist. de Fréchet entre features de Inception | Fidelidad + diversidad |
| **IS** (Inception Score) | $\exp(\mathbb{E}[\text{KL}(p(y|x)\|p(y))])$ | Calidad + diversidad |
| **Precision/Recall** | Cobertura de modos reales | Fidelidad vs diversidad |

### Para datos tabulares

| Métrica | Descripción |
|---------|-------------|
| **Estadísticas marginales** | Media, std, cuantiles por columna |
| **Correlación de columnas** | Diferencia de matrices de correlación |
| **Machine Learning Efficacy** | Entrenar en sintético, evaluar en real |
| **Privacy metrics** | Distance to closest record |

### FID (Fréchet Inception Distance)

$$\text{FID} = \|\mu_r - \mu_g\|^2 + \text{Tr}(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2})$$

Un FID menor es mejor. FID=0 implica distribuciones idénticas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm

np.random.seed(7)

def frechet_distance(mu1, sigma1, mu2, sigma2):
    # FID generalizado (sin Inception network)
    diff = mu1 - mu2
    covmean = sqrtm(sigma1 @ sigma2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2*covmean)
    return float(fid)

def compute_stats(X):
    return X.mean(axis=0), np.cov(X.T)

def mle_efficacy(X_train_synthetic, y_train, X_test_real, y_test):
    # Entrenar en sintético, evaluar en real (Train-Synth-Test-Real)
    from collections import Counter
    majority_class = Counter(y_train).most_common(1)[0][0]
    # Clasificador simple: centroide más cercano
    classes = list(set(y_train))
    centroids = {c: X_train_synthetic[y_train==c].mean(0) for c in classes}
    preds = []
    for x in X_test_real:
        dists = {c: np.linalg.norm(x - centroids[c]) for c in classes}
        preds.append(min(dists, key=dists.get))
    acc = np.mean(np.array(preds) == y_test)
    return acc

# Comparar modelos: GAN final, VAE final, KDE, SMOTE
x_real_eval  = sample_real(1000)
x_gan_eval   = gan.generate(1000)
x_vae_eval   = vae.generate(1000)
x_kde_eval   = gaussian_kde(x_real_eval.T).resample(1000).T

# Mode collapse simulation (sólo 1 de 3 gaussianas)
x_collapsed  = np.random.randn(1000, 2)*0.5 + [2, 2]

mu_real, cov_real = compute_stats(x_real_eval)

models = {
    'GAN (entrenado)': x_gan_eval,
    'VAE (entrenado)': x_vae_eval,
    'KDE':             x_kde_eval,
    'GAN (collapse)':  x_collapsed,
}

print(f'{"Modelo":<20} | {"FID":>8} | {"ΔMean":>8} | {"ΔStd":>8}')
print('-' * 55)

fids, names = [], []
for name, X_gen in models.items():
    mu_gen, cov_gen = compute_stats(X_gen)
    fid = frechet_distance(mu_real, cov_real, mu_gen, cov_gen)
    delta_mean = np.linalg.norm(mu_real - mu_gen)
    delta_std  = np.abs(np.sqrt(np.diag(cov_real)) - np.sqrt(np.diag(cov_gen))).mean()
    print(f'{name:<20} | {fid:>8.4f} | {delta_mean:>8.4f} | {delta_std:>8.4f}')
    fids.append(fid); names.append(name)

# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_comp = ['#2ECC71', '#9B59B6', '#F39C12', '#E74C3C']
bars = axes[0].bar(range(len(names)), fids, color=colors_comp, alpha=0.85, edgecolor='white', linewidth=2)
for bar, fid in zip(bars, fids):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{fid:.2f}', ha='center', fontsize=12, fontweight='bold')
axes[0].set_xticks(range(len(names)))
axes[0].set_xticklabels(names, rotation=15, fontsize=10)
axes[0].set_ylabel('FID (menor = mejor)', fontsize=12)
axes[0].set_title('Fréchet Distance por modelo', fontsize=13, fontweight='bold')
axes[0].grid(True, axis='y', alpha=0.3)

# Scatter comparativo
axes[1].scatter(x_real_eval[:,0], x_real_eval[:,1], s=6, c='gray', alpha=0.3, label='Real')
for (name, X_gen), color in zip(models.items(), colors_comp):
    axes[1].scatter(X_gen[:100,0], X_gen[:100,1], s=12, c=color, alpha=0.7, label=name)
axes[1].set_title('Comparación visual de distribuciones', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9, loc='upper right'); axes[1].grid(True, alpha=0.2)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

## 8.2 Tabla resumen de todos los modelos

| Modelo | Tipo | $p(x)$ exacta | Velocidad muestreo | Datos tabulares | Calidad imagen | Complejidad impl. |
|--------|------|:-------------:|:-----------------:|:---------------:|:--------------:|:-----------------:|
| **GAN** | Implícito | ✗ | ⚡⚡⚡ | Medio | Alta | Alta |
| **WGAN-GP** | Implícito | ✗ | ⚡⚡⚡ | Medio | Muy alta | Alta |
| **VAE** | Explícito aprox. | ELBO | ⚡⚡⚡ | Bueno | Media | Media |
| **Norm. Flow** | Explícito exacto | ✓ | ⚡⚡⚡ | Bueno | Alta | Alta |
| **DDPM** | Explícito aprox. | ELBO | ⚡ (lento) | Emergente | State-of-art | Muy alta |
| **CTGAN** | GAN tabular | ✗ | ⚡⚡⚡ | **Excelente** | N/A | Media |
| **Cópula Gaussiana** | Estadístico | ✓ | ⚡⚡⚡ | **Excelente** | N/A | Baja |
| **KDE** | No paramétrico | ✓ | ⚡⚡ | Aceptable | N/A | Muy baja |
| **SMOTE** | Interpolación | ✗ | ⚡⚡⚡ | Solo sobre-muestreo | N/A | Muy baja |

## 8.3 ¿Cuándo usar cada modelo?

```
¿Qué tipo de dato?
├── Imágenes / audio
│   ├── ¿Necesitas máxima calidad y tienes recursos?  → Stable Diffusion / DDPM
│   ├── ¿Velocidad de entrenamiento importa?          → StyleGAN3 / BigGAN
│   └── ¿Quieres log-likelihood exacta?               → Normalizing Flow
│
└── Datos tabulares
    ├── ¿Mezcla de tipos numéricos+categóricos?       → CTGAN / TabDDPM
    ├── ¿Preservar correlaciones estadísticas?        → Cópula Gaussiana
    ├── ¿Oversampling para imbalance?                 → SMOTE / ADASYN
    ├── ¿Dataset pequeño, distribución simple?        → KDE
    └── ¿Privacidad y utilidad simultáneas?           → DP-CTGAN / PATE-GAN
```